# 09 AdaBoost

依赖安装说明：`pip install numpy matplotlib scikit-learn`

AdaBoost 是 boosting 的经典算法。它会连续训练很多弱分类器，每一轮更关注上一轮分错的样本。


## 1. 数学逻辑

AdaBoost 维护样本权重 `D_i`。弱分类器错误率：

$$err = \sum_i D_i \cdot I(h(x_i) \ne y_i)$$

弱分类器权重：

$$\alpha = \frac{1}{2}\log\frac{1-err}{err}$$

分错的样本权重会变大，分对的会变小。最终预测是加权投票：

$$H(x)=\text{sign}\left(\sum_t \alpha_t h_t(x)\right)$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

np.random.seed(42)
X, y01 = make_classification(n_samples=250, n_features=2, n_redundant=0, n_informative=2,
                             n_clusters_per_class=1, class_sep=0.9, random_state=42)
y = np.where(y01 == 1, 1, -1)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)


In [ ]:
# 从零实现：用一维阈值 stump 做弱分类器

def fit_stump(X, y, sample_weight):
    best = {'err': float('inf')}
    for feature in range(X.shape[1]):
        for threshold in np.unique(X[:, feature]):
            for polarity in [1, -1]:
                pred = np.ones(len(y))
                pred[polarity * X[:, feature] < polarity * threshold] = -1
                err = np.sum(sample_weight[pred != y])
                if err < best['err']:
                    best = {'feature': feature, 'threshold': threshold, 'polarity': polarity, 'err': err, 'pred': pred}
    return best

sample_weight = np.ones(len(X_train)) / len(X_train)
learners = []
for t in range(12):
    stump = fit_stump(X_train, y_train, sample_weight)
    err = np.clip(stump['err'], 1e-12, 1 - 1e-12)
    alpha = 0.5 * np.log((1 - err) / err)
    sample_weight *= np.exp(-alpha * y_train * stump['pred'])
    sample_weight /= sample_weight.sum()
    learners.append((stump, alpha))
    print(f'round {t+1:2d} | err={err:.3f} | alpha={alpha:.3f}')


In [ ]:
base = DecisionTreeClassifier(max_depth=1, random_state=42)
try:
    model = AdaBoostClassifier(estimator=base, n_estimators=60, learning_rate=0.8, random_state=42)
except TypeError:
    model = AdaBoostClassifier(base_estimator=base, n_estimators=60, learning_rate=0.8, random_state=42)

model.fit(X_train, y_train)
pred = model.predict(X_test)
print('AdaBoost accuracy:', round(accuracy_score(y_test, pred), 3))

xx, yy = np.meshgrid(np.linspace(X[:,0].min()-0.5, X[:,0].max()+0.5, 180),
                     np.linspace(X[:,1].min()-0.5, X[:,1].max()+0.5, 180))
grid = np.c_[xx.ravel(), yy.ravel()]
zz = model.predict(grid).reshape(xx.shape)
plt.contourf(xx, yy, zz, alpha=0.25, cmap='coolwarm')
plt.scatter(X_train[:,0], X_train[:,1], c=y_train, cmap='coolwarm', edgecolor='k', s=24)
plt.title('AdaBoost 决策边界')
plt.show()


## 2. 常见误区

- AdaBoost 对异常值和错误标签比较敏感，因为它会越来越关注难分样本。
- 弱学习器通常要弱，常见选择是深度为 1 的决策树 stump。
- `learning_rate` 和 `n_estimators` 需要一起调。

## 3. 小实验

- 增加 `n_estimators`，观察训练是否过拟合。
- 调低 `learning_rate`，通常需要更多轮。
- 把弱学习器深度从 1 改到 2，观察边界变化。
